In [1]:
import pandas as pd
import numpy as np
from nltk import word_tokenize
import os
from ast import literal_eval
from nltk.util import ngrams
from nltk.corpus import stopwords
stop=stopwords.words('english')
import spacy
nlp=spacy.load('en_core_web_sm')
from tqdm import tqdm

/Users/mstudio/miniconda3/envs/py3.10/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
path='/Users/mstudio/Library/CloudStorage/Box-Box/coolie/sent/'
df=pd.read_csv(path+'/coolie.csv', converters={'state':literal_eval, 'context':literal_eval, 'city':literal_eval})

In [3]:
df.columns

Index(['state', 'city', 'date', 'lccn', 'sent', 'title', 'context'], dtype='object')

In [14]:
def get_ngrams(text, n):
    entire=[]
    n_grams=ngrams(word_tokenize(text), n)
    for i in n_grams:
        entire.append(i)
    return entire

In [3]:
df['context_str']=df['context'].apply(' '.join)

In [4]:
def lemmatization(dataframe:pd.DataFrame()):
    dataframe['punct_word']=dataframe['sent'].apply(lambda row: ' '.join([w.lemma_ for w in nlp(row)]))
    dataframe['sent_stop']=dataframe['punct_word'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))
    dataframe['sent_punct']=dataframe['sent_stop'].str.replace('[^\w\s]','')
    dataframe['sent_lemma']=dataframe['sent_punct'].apply(lambda x: ' '.join(x for x in x.split() if x.isalpha()))
    dataframe['sent_token']=dataframe['sent_lemma'].apply(word_tokenize)
    dataframe.drop(['sent_stop', 'sent_punct', 'punct_word'], axis=1, inplace=True)
    return dataframe

In [20]:
def reuse_df(data:pd.DataFrame()):
    no_reuse=[]
    pair_index1=[]
    pair_index2=[]
    pair_date1=[]
    pair_date2=[]
    pair_text1=[]
    pair_text2=[]
    pair_state1=[]
    pair_state2=[]
    for k in range(0,data.shape[0]):
        t=0
        for p in range(t,data.shape[0]):
            t+=1
            if t>k:
                if (k==p) & (len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))) == len(data['5-gram'][k])):
                    pass
                elif len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))) == 0:
                    pass
                elif len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))) >= 3:
                    no_reuse.append(len(list(set(data['5-gram'][k]).intersection(data['5-gram'][p]))))
                    pair_index1.append(k) 
                    pair_index2.append(p)
                    pair_date1.append(data.iloc[k]['date'])
                    pair_date2.append(data.iloc[p]['date'])
                    pair_text1.append(data.iloc[k]['lemma'])
                    pair_text2.append(data.iloc[p]['lemma'])
                    pair_state1.append(data.iloc[k]['state'])
                    pair_state2.append(data.iloc[p]['state'])
    return pd.DataFrame({'no_reuse':no_reuse, 'pair_index1':pair_index1, 'pair_index2':pair_index2, 
             'pair_date1':pair_date1, 'pair_date2':pair_date2, 
              'pair_text1':pair_text1, 'pair_text2':pair_text2,
                'pair_state1':pair_state1, 'pair_state2':pair_state2})

In [5]:
df=lemmatization(df)

In [6]:
df

,state,city,date,lccn,sent,title,context,context_str,sent_lemma,sent_token
0,[New York],[New York],18910826,sn83030193,cholera breaks out on a steamer\nladen with oh...,The evening world. [volume],"[sessions., m m\ncruise oe a death ship., chol...",sessions. m m\ncruise oe a death ship. cholera...,cholera break steamer laden ohinoso coolie,"[cholera, break, steamer, laden, ohinoso, coolie]"
1,[New York],[New York],18910826,sn83030193,atlt ices from\nslngnporo state that tho steam...,The evening world. [volume],"[san i'iuxcisco, aug., !'!!., atlt ices from\n...","san i'iuxcisco, aug. !'!!. atlt ices from\nsln...",atlt ice slngnporo state tho steamer namchow s...,"[atlt, ice, slngnporo, state, tho, steamer, na..."
2,[Montana],[Butte],18830407,sn84036033,"therefore, it is not\nsurprising that the auth...",The semi-weekly miner. [volume],[at present the miserable\nkanakas are but one...,at present the miserable\nkanakas are but one ...,therefore surprising authority kalakaua kingdo...,"[therefore, surprising, authority, kalakaua, k..."
3,[Montana],[Butte],18830407,sn84036033,kanakas want coolies.,The semi-weekly miner. [volume],"[great excitement prevails, and it., is\nbelie...","great excitement prevails, and it. is\nbelieve...",kanakas want coolie,"[kanakas, want, coolie]"
4,[Montana],[Butte],18851114,sn84036033,standing by the coolies.,The semi-weekly miner. [volume],"[he appoints\nwilliam c. prime, of this city, ...","he appoints\nwilliam c. prime, of this city, h...",stand coolie,"[stand, coolie]"
...,...,...,...,...,...,...,...,...,...,...
118049,[Maryland],[Thurmont],19200108,sn84026688,most of this agitation yame from the\n■ongue a...,Catoctin clarion. [volume],[(' the use of model'll methods in the\nfar ea...,(' the use of model'll methods in the\nfar eas...,agitation yame ongue pen thei educate chi nese...,"[agitation, yame, ongue, pen, thei, educate, c..."
118050,[New York],[New York],19111029,sn83030272,with porters and eighty coolies.,The sun. [volume],"[mrs workman ' italian j tions., , guid., with...","mrs workman ' italian j tions. , guid. with po...",porter eighty coolie,"[porter, eighty, coolie]"
118051,[District of Columbia],[Washington],19251015,sn83045462,"r,\nmany persons are fleeing into thrf\nforeig...",Evening star. [volume],[the purpose-of this move-4\nment is to instit...,the purpose-of this move-4\nment is to institu...,r many person flee thrf foreign concession bri...,"[r, many, person, flee, thrf, foreign, concess..."
118052,[District of Columbia],[Washington],19250713,sn83045462,—reports\nfrom changsha say a strike of coolie...,Evening star. [volume],"[hankow., july 13 (/p)., —reports\nfrom changs...",hankow. july 13 (/p). —reports\nfrom changsha ...,report changsha say strike coolie begin friday...,"[report, changsha, say, strike, coolie, begin,..."


In [11]:
df.to_csv(path+'coolie-lemma-df-sent2.csv', index=False)

In [12]:
df=df.rename(columns={'snet_token':'sent_token'})

In [9]:
df.drop(['context_str'], axis=1, inplace=True)

In [10]:
df.columns

Index(['state', 'city', 'date', 'lccn', 'sent', 'title', 'context',
       'sent_lemma', 'sent_token'],
      dtype='object')

In [12]:
path='/Users/mstudio/Library/CloudStorage/Box-Box/coolie/sent/'
df=pd.read_csv(path+'/coolie-lemma.csv', converters={'state':literal_eval, 'context':literal_eval, 'city':literal_eval})

In [13]:
df.shape

(316806, 11)

In [15]:
df['5-gram']=df['context_str'].apply(lambda x: get_ngrams(x, 5))

In [21]:
reprintdf=reuse_df(df)

KeyboardInterrupt: 

In [ ]:
reprintdf.to_csv(path+'reprint.csv', index=False)